In [3]:
#!pip install optuna

In [8]:
# ============================
# OTIMIZAÇÃO COM OPTUNA + HOLDOUT + CV INTERNA
# ============================
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, classification_report

def optuna_tuning_holdout_cv(model_name, X_train, y_train, n_trials=60):
    def objective(trial):
        # Definindo hiperparâmetros
        if model_name == "Regressão Logística":
            params = {
                "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
                "solver": trial.suggest_categorical("solver", ["lbfgs", "liblinear"]),
                "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"])
            }
            model = LogisticRegression(max_iter=1000, **params)

        elif model_name == "MLP":
            params = {
                "hidden_layer_sizes": trial.suggest_categorical(
                    "hidden_layer_sizes",
                    [(50,), (100,), (200,), (100,50), (150,100), (200,100), (100,100)]
                ),
                "activation": trial.suggest_categorical("activation", ["relu", "tanh", "logistic"]),
                "alpha": trial.suggest_float("alpha", 1e-5, 1e-2, log=True),
                "learning_rate_init": trial.suggest_float("learning_rate_init", 5e-4, 1e-2, log=True)
            }
            model = MLPClassifier(max_iter=1000, random_state=37, **params)

        else:  # Naive Bayes
            model = GaussianNB()
            params = {}

        # Pipeline com escalonamento e SMOTE
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=37)),
            ('model', model)
        ])

        # CV estratificada no treino
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=37)
        f1 = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1).mean()
        return f1

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)
    return study.best_params, study.best_value

# ============================
# EXECUÇÃO
# ============================
modelos = ["Regressão Logística", "MLP", "Naive Bayes"]
resultados = {}

for m in modelos:
    print("\n======================================")
    print(f"Otimização Optuna – {m}")
    best_params, best_score = optuna_tuning_holdout_cv(m, X_train, y_train, n_trials=60)
    print("Melhores hiperparâmetros:", best_params)
    print("F1 Médio na CV interna:", best_score)

    # Treinar modelo final no conjunto completo de treino
    if m == "Regressão Logística":
        final_model = LogisticRegression(max_iter=1000, **best_params)
    elif m == "MLP":
        final_model = MLPClassifier(max_iter=1000, random_state=37, **best_params)
    else:
        final_model = GaussianNB()

    pipeline_final = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=37)),
        ('model', final_model)
    ])

    pipeline_final.fit(X_train, y_train)
    y_pred = pipeline_final.predict(X_test)
    f1_test = f1_score(y_test, y_pred)

    resultados[m] = {
        "Modelo": pipeline_final,
        "F1 Teste": f1_test,
        "Parâmetros": best_params
    }

    print(f"\nF1 Teste - {m}: {f1_test:.4f}")
    print(classification_report(y_test, y_pred))

[I 2025-12-08 17:41:23,446] A new study created in memory with name: no-name-2458560d-844a-4ee0-87fc-6bcd3daf92a5



Otimização Optuna – Regressão Logística


[I 2025-12-08 17:41:23,732] Trial 0 finished with value: 0.7712682912320598 and parameters: {'C': 4.098612076228079, 'solver': 'lbfgs', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7712682912320598.
[I 2025-12-08 17:41:24,060] Trial 1 finished with value: 0.7712682912320598 and parameters: {'C': 0.787596415006529, 'solver': 'lbfgs', 'class_weight': None}. Best is trial 0 with value: 0.7712682912320598.
[I 2025-12-08 17:41:24,763] Trial 2 finished with value: 0.7664575784828078 and parameters: {'C': 0.04803138066113974, 'solver': 'lbfgs', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7712682912320598.
[I 2025-12-08 17:41:25,067] Trial 3 finished with value: 0.7493259904727299 and parameters: {'C': 0.0028592146672124633, 'solver': 'liblinear', 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7712682912320598.
[I 2025-12-08 17:41:25,311] Trial 4 finished with value: 0.770717181718589 and parameters: {'C': 0.17713658023263434, 'solver': 'liblinear', 'cl

Melhores hiperparâmetros: {'C': 4.098612076228079, 'solver': 'lbfgs', 'class_weight': 'balanced'}
F1 Médio na CV interna: 0.7712682912320598

F1 Teste - Regressão Logística: 0.7862
              precision    recall  f1-score   support

           0       0.55      0.69      0.61       196
           1       0.84      0.74      0.79       431

    accuracy                           0.72       627
   macro avg       0.69      0.72      0.70       627
weighted avg       0.75      0.72      0.73       627


Otimização Optuna – MLP


[I 2025-12-08 17:42:09,851] Trial 0 finished with value: 0.7699006361270985 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'logistic', 'alpha': 3.2575048113255004e-05, 'learning_rate_init': 0.004052833791926637}. Best is trial 0 with value: 0.7699006361270985.
/usr/local/lib/python3.12/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (50,) which is of type tuple.
  warnings.warn(message)
/usr/local/lib/python3.12/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains (100,) which is of type tuple.
  warnings.warn(message)
/usr/local/lib/python3.12/dist-packages/optuna/distributions.py:502: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage b

Melhores hiperparâmetros: {'hidden_layer_sizes': (150, 100), 'activation': 'tanh', 'alpha': 3.2122235323260364e-05, 'learning_rate_init': 0.0007254376977849637}
F1 Médio na CV interna: 0.8092004600132598


[I 2025-12-08 19:13:27,229] A new study created in memory with name: no-name-cd037a3e-149f-489f-92f0-fe0449561223
[I 2025-12-08 19:13:27,408] Trial 0 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.



F1 Teste - MLP: 0.8074
              precision    recall  f1-score   support

           0       0.58      0.58      0.58       196
           1       0.81      0.81      0.81       431

    accuracy                           0.74       627
   macro avg       0.69      0.69      0.69       627
weighted avg       0.74      0.74      0.74       627


Otimização Optuna – Naive Bayes


[I 2025-12-08 19:13:27,535] Trial 1 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:27,650] Trial 2 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:27,767] Trial 3 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:27,882] Trial 4 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:27,998] Trial 5 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:28,115] Trial 6 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:28,241] Trial 7 finished with value: 0.74612026717859 and parameters: {}. Best is trial 0 with value: 0.74612026717859.
[I 2025-12-08 19:13:

Melhores hiperparâmetros: {}
F1 Médio na CV interna: 0.74612026717859

F1 Teste - Naive Bayes: 0.7526
              precision    recall  f1-score   support

           0       0.51      0.73      0.60       196
           1       0.85      0.68      0.75       431

    accuracy                           0.69       627
   macro avg       0.68      0.70      0.68       627
weighted avg       0.74      0.69      0.70       627

